# AXE 2 — 02. ALS Model
_Notebook 2/2 — Entraînement ALS + scoring → `warehouse/als_scores.parquet`_

## Approche
Les interactions sont déjà au format multi-users (chaque mois = un virtual user).  
ALS entraîne une vraie décomposition matricielle (users × items).  
Pour scorer chaque item :

1. On extrait les **item factors** (vecteurs latents de dimension `rank`)
2. On calcule un **vecteur de référence** = moyenne des items les plus consommés
3. **Score** = similarité cosinus (item factor vs vecteur de référence) → 0–100%

**Output** : `warehouse/als_scores.parquet`

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
# Si ce notebook plante avec "Connection refused" ou "no resources" :
#   → Kernel > Restart Kernel and Clear Outputs, puis relancer depuis ici.
# NE PAS lancer ce notebook dans le même kernel qu'un autre notebook Spark.

import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Model") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"App : {spark.sparkContext.appName}")

WAREHOUSE = "/opt/spark/warehouse" if os.path.exists("/opt/spark/warehouse") \
            else "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(os.path.join(WAREHOUSE, "interactions")), \
    "Lance d'abord 01_build_interactions.ipynb"

Spark version : 3.5.5
App : PySparkShell
Warehouse: /opt/spark/warehouse


26/04/06 13:24:40 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# ── 1. CHARGEMENT ─────────────────────────────────────────────────────────────
interactions = spark.read.parquet(os.path.join(WAREHOUSE, "interactions"))

n_users = interactions.select("user_id").distinct().count()
n_items = interactions.select("item_id").distinct().count()
n_rows  = interactions.count()

print(f"Interactions : {n_rows:,}")
print(f"Virtual users (mois) : {n_users}")
print(f"Items distincts : {n_items:,}")
interactions.printSchema()
interactions.groupBy("platform").count().orderBy("platform").show()

Interactions : 17,340
Virtual users (mois) : 82
Items distincts : 4,580
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- item_title: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- play_count: long (nullable = true)

+--------+-----+
|platform|count|
+--------+-----+
| netflix|  267|
| spotify|15353|
| youtube| 1720|
+--------+-----+



In [3]:
# ── 2. CAST play_count en double pour ALS ─────────────────────────────────────
# ALS attend une colonne de rating numérique

interactions = interactions.withColumn("weight", F.col("play_count").cast("double"))
interactions.select("user_id", "item_id", "weight").show(5)

+-------+-------+------+
|user_id|item_id|weight|
+-------+-------+------+
| 202508|    719|   5.0|
| 202508|   1546|   1.0|
| 202508|    230|   2.0|
| 202508|   2007|   3.0|
| 202508|    316|  10.0|
+-------+-------+------+
only showing top 5 rows



In [4]:
# ── 3. TRAIN / TEST SPLIT (80/20) ─────────────────────────────────────────────
train, test = interactions.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {train.count():,}  |  Test : {test.count():,}")

Train : 13,932  |  Test : 3,408


In [5]:
# ── 4. ENTRAÎNEMENT ALS ───────────────────────────────────────────────────────
# rank=20 (au lieu de 50) et maxIter=10 pour éviter l'OOM dans Docker.
# Avec ~80 virtual users et quelques milliers d'items, rank=20 est largement suffisant.

from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id",
    itemCol="item_id",
    ratingCol="weight",
    rank=20,
    maxIter=10,
    regParam=0.1,
    implicitPrefs=True,
    coldStartStrategy="drop",
    seed=42
)

print("Training ALS (rank=20, maxIter=10, implicitPrefs=True)...")
model = als.fit(train)
print("Done.")

Training ALS (rank=20, maxIter=10, implicitPrefs=True)...


26/04/06 13:25:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Done.


In [6]:
# ── 5. ÉVALUATION ─────────────────────────────────────────────────────────────
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="weight",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"RMSE (test set) : {rmse:.4f}")

RMSE (test set) : 3.1657


In [7]:
# ── 6. ITEM FACTORS → VECTEUR DE RÉFÉRENCE ────────────────────────────────────
# Les item factors encodent les patterns de co-consommation mensuelle
# On calcule un vecteur de référence = moyenne des top-N items les plus consommés
# Score d'un item = cosine similarity(item factor, vecteur de référence)

item_factors_df = model.itemFactors.toPandas()
item_factors_df = item_factors_df.rename(columns={"id": "item_id", "features": "factors"})
print(f"Item factors : {len(item_factors_df)} items, rank={len(item_factors_df['factors'].iloc[0])}")

# Matrice de factors
factor_matrix = np.vstack(item_factors_df["factors"].values)  # (n_items, rank)
ids_array     = item_factors_df["item_id"].values

# Normalisation L2
norms = np.linalg.norm(factor_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-8
factor_norm = factor_matrix / norms

print(f"Factor matrix : {factor_matrix.shape}")

Item factors : 4512 items, rank=20
Factor matrix : (4512, 20)


In [8]:
# ── 7. CALCUL DES SCORES ──────────────────────────────────────────────────────
# total_weight par item = somme des play_count sur tous les mois

item_totals = interactions.groupBy("item_id", "item_title", "platform").agg(
    F.sum("play_count").alias("total_plays")
).toPandas()

print(f"Items total : {len(item_totals)}")

# Top 50 items les plus consommés → vecteur de référence
top50 = item_totals.sort_values("total_plays", ascending=False).head(50)
top_ids = set(top50["item_id"].values)

id_to_idx = {int(id_): idx for idx, id_ in enumerate(ids_array)}

top_indices = [id_to_idx[int(i)] for i in top_ids if int(i) in id_to_idx]
if not top_indices:
    raise ValueError("Aucun item des top-50 trouvé dans les item factors. Vérifie l'alignement item_id.")

ref_vector = factor_norm[top_indices].mean(axis=0)
ref_vector = ref_vector / (np.linalg.norm(ref_vector) + 1e-8)

# Cosine similarity de tous les items vs référence
similarities = factor_norm @ ref_vector  # (n_items,)

# Normalisation → 0–100
s_min, s_max = similarities.min(), similarities.max()
if s_max > s_min:
    scores = (similarities - s_min) / (s_max - s_min) * 100
else:
    scores = np.full_like(similarities, 50.0)

scores_df = pd.DataFrame({"item_id": ids_array.astype(int), "predicted_score": scores.round(1)})
print(f"Scores : min={scores.min():.1f} max={scores.max():.1f} mean={scores.mean():.1f}")

Items total : 4580
Scores : min=0.0 max=100.0 mean=34.5


In [9]:
# ── 8. ASSEMBLAGE ─────────────────────────────────────────────────────────────

item_totals["item_id"] = item_totals["item_id"].astype(int)
result_df = item_totals.merge(scores_df, on="item_id", how="inner")
result_df = result_df.sort_values("predicted_score", ascending=False).reset_index(drop=True)
result_df["rank"] = result_df["predicted_score"].rank(ascending=False, method="first").astype(int)

print(f"Résultat final : {len(result_df)} items")
print("\nTop 20 :")
print(result_df[["rank", "platform", "item_title", "predicted_score", "total_plays"]].head(20).to_string(index=False))

Résultat final : 4512 items

Top 20 :
 rank platform                                              item_title  predicted_score  total_plays
    1  spotify                                       Radiohead — Creep            100.0           32
    2  spotify                                        Kalash — Tombolo             97.9           23
    3  spotify                                Luv Resval — Cette fille             96.3           24
    4  spotify                                    Tiakola — Pousse toi             95.5           17
    5  spotify                                              MHD — Eyla             94.8           22
    6  spotify                                         Kalash — Laptop             94.4           25
    7  spotify               Guy2Bezbar — Jerrican (feat. La Mano 1.9)             93.9           71
    8  spotify                                       KeBlack — Zizanie             93.6           27
    9  spotify                         Tiakola — Gaso

In [10]:
# ── 9. ÉCRITURE warehouse/als_scores ──────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

schema = StructType([
    StructField("item_id",         IntegerType(), True),
    StructField("item_title",      StringType(),  True),
    StructField("platform",        StringType(),  True),
    StructField("play_count",      IntegerType(), True),
    StructField("predicted_score", FloatType(),   True),
    StructField("rank",            IntegerType(), True),
])

result_df = result_df.rename(columns={"total_plays": "play_count"})
result_df["play_count"]      = result_df["play_count"].astype(int)
result_df["predicted_score"] = result_df["predicted_score"].astype(float)
result_df["rank"]            = result_df["rank"].astype(int)

out_spark = spark.createDataFrame(
    result_df[["item_id", "item_title", "platform", "play_count", "predicted_score", "rank"]],
    schema=schema
)

out_path = os.path.join(WAREHOUSE, "als_scores")
out_spark.write.mode("overwrite").parquet(out_path)

print(f"Écrit : {out_path}")
spark.read.parquet(out_path).orderBy(F.desc("predicted_score")).show(10, truncate=55)

spark.stop()
print("Notebook 02 terminé. Le dashboard /recommandations est maintenant actif.")

Écrit : /opt/spark/warehouse/als_scores
+-------+-----------------------------------------+--------+----------+---------------+----+
|item_id|                               item_title|platform|play_count|predicted_score|rank|
+-------+-----------------------------------------+--------+----------+---------------+----+
|     19|                        Radiohead — Creep| spotify|        32|          100.0|   1|
|     12|                         Kalash — Tombolo| spotify|        23|           97.9|   2|
|      3|                 Luv Resval — Cette fille| spotify|        24|           96.3|   3|
|     75|                     Tiakola — Pousse toi| spotify|        17|           95.5|   4|
|     60|                               MHD — Eyla| spotify|        22|           94.8|   5|
|     11|                          Kalash — Laptop| spotify|        25|           94.4|   6|
|     45|Guy2Bezbar — Jerrican (feat. La Mano 1.9)| spotify|        71|           93.9|   7|
|     13|                     